# 02 — Models

Sanity-checks all 8 models from `models/`. For each model:
1. Simulate 1 agent × 80 trials with sensible default parameters
2. Verify log-likelihood: true params should score higher than random params
3. Plot cumulative reward over trials

At the end, compare reward curves across all models.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

from env import Env
from models import ALL_MODELS, MF, SF, MB, MFP, SFP, RRMF, RRSF, RRMB
from models.base import DEFAULT_PI0, pack, unpack


In [ ]:
# ── Shared setup ──────────────────────────────────────────────────────────────
RNG  = np.random.default_rng(0)
PI0  = DEFAULT_PI0   # uniform default policy (used for pi0_init)
TRIALS = Env.make_trial_sequence(seed=0)

# Default parameters for each model — reasonable mid-range values
DEFAULT_PARAMS = {
    'MF':   {'gamma': 0.9, 'alpha': 0.3,  'tau': 1.0},
    'SF':   {'gamma': 0.9, 'alpha_sf': 0.3, 'tau': 1.0},
    'MB':   {'gamma': 0.9, 'tau': 1.0,    'alpha_t': 0.3, 'alpha_r': 0.3},
    'MFP':  {'gamma': 0.9, 'alpha': 0.3,  'tau': 1.0, 'alpha_m': 0.3, 'h': 1.0, 'lam': 0.05},
    'SFP':  {'gamma': 0.9, 'alpha_sf': 0.3, 'tau': 1.0, 'alpha_m': 0.3, 'h': 1.0, 'lam': 0.05},
    'RRMF': {'beta': 2.0,  'gamma': 0.9,  'alpha': 0.3, 'alpha_p': 0.1, 'lam': 0.05},
    'RRSF': {'beta': 2.0,  'gamma': 0.9,  'alpha_sf': 0.3, 'alpha_p': 0.1, 'lam': 0.05},
    'RRMB': {'beta': 2.0,  'gamma': 0.9,  'alpha_t': 0.3, 'alpha_r': 0.3},
}

MODULES = ALL_MODELS

print('Setup complete. Trial sequence length:', len(TRIALS))
print('Goal distribution:', {g: TRIALS.count(g) for g in Env.TRAINING_GOALS})

## Unit tests for base utilities

In [ ]:
from models.base import softmax, rr_policy

# Test 1: softmax of zeros = uniform
assert np.allclose(softmax([0., 0., 0.], tau=1.0), [1/3, 1/3, 1/3]), 'softmax zeros failed'

# Test 2: rr_policy with Q=zeros → returns π_0 exactly
p0 = np.array([0.5, 0.3, 0.2])
assert np.allclose(rr_policy([0., 0., 0.], p0, beta=1.0), p0), 'rr_policy Q=0 failed'

# Test 3: rr_policy with uniform π_0 → reduces to softmax
q   = np.array([1.0, 2.0, 0.5])
p0u = np.ones(3) / 3
assert np.allclose(rr_policy(q, p0u, beta=1.0), softmax(q, tau=1.0)), 'rr_policy uniform π_0 failed'

print('All base utility tests passed ✓')

## Simulate all models + log-likelihood check

For each model, we simulate one agent and verify that the true parameters achieve **higher log-likelihood** on the simulated data than 10 random parameter sets.  
This tests both that `simulate` and `log_likelihood` are internally consistent.

In [ ]:
def random_params(spec, rng):
    """Sample random parameters in natural space from uniform over reasonable ranges."""
    p = {}
    for name, tr in spec:
        if   tr == 'logit': p[name] = float(rng.uniform(0.05, 0.95))
        elif tr == 'log':   p[name] = float(rng.uniform(0.1, 5.0))
        else:               p[name] = float(rng.uniform(0.0, 1.0))
    return p


results = {}

for name, mod in MODULES.items():
    params = DEFAULT_PARAMS[name]
    rng_local = np.random.default_rng(42)

    # Simulate
    actions = mod.simulate(TRIALS, params, PI0, rng_local)

    # LL under true params
    ll_true = mod.log_likelihood(actions, TRIALS, params, PI0)

    # LL under random params
    ll_random = []
    for _ in range(10):
        rp = random_params(mod.PARAM_SPEC, rng_local)
        ll_random.append(mod.log_likelihood(actions, TRIALS, rp, PI0))

    n_better = sum(ll_true > lr for lr in ll_random)
    status   = '✓' if n_better >= 7 else f'⚠ only {n_better}/10'

    # Rewards
    rewards = [Env.reward(Env.step(Env.step(0, a[0]), a[1]), Env.GOALS[g])
               for a, g in zip(actions, TRIALS)]

    results[name] = {'actions': actions, 'll_true': ll_true,
                     'll_random': ll_random, 'rewards': rewards}

    print(f'{name:6s}  LL_true={ll_true:7.1f}  LL_rand_mean={np.mean(ll_random):7.1f}  '
          f'true > rand {n_better}/10  {status}')

## Reward curves per model

In [ ]:
def smooth(x, w=10):
    """Simple moving average."""
    return np.convolve(x, np.ones(w) / w, mode='valid')

fig = go.Figure()
colors = px.colors.qualitative.Plotly

for i, (name, res) in enumerate(results.items()):
    r_smooth = smooth(res['rewards'])
    fig.add_trace(go.Scatter(
        x=list(range(len(r_smooth))),
        y=r_smooth,
        name=name,
        line=dict(color=colors[i]),
    ))

fig.update_layout(
    title='Smoothed reward per trial (window=10), all models',
    xaxis_title='Trial',
    yaxis_title='Reward (moving avg)',
    width=750, height=400,
)
fig.show()

## Per-model detail: action distributions over trials

In [ ]:
def plot_model_detail(name, res):
    actions = res['actions']
    rewards = res['rewards']

    a0s = [a[0] for a in actions]   # stage-1 choices: 0=left, 1=mid, 2=right
    a1s = [a[1] for a in actions]   # stage-2 choices: 0=left, 1=right

    # Terminal rooms visited
    rooms = [Env.step(Env.step(0, a[0]), a[1]) for a in actions]

    fig = make_subplots(rows=1, cols=3,
                        subplot_titles=['Stage-1 action freq',
                                        'Terminal room freq',
                                        'Cumulative reward'])

    # Stage-1 action histogram
    a0_counts = [a0s.count(i) for i in range(3)]
    fig.add_trace(go.Bar(x=['left', 'mid', 'right'], y=a0_counts,
                         marker_color=['#4a90d9', '#7bc67b', '#e07b39']),
                  row=1, col=1)

    # Room histogram
    room_counts = [rooms.count(s) for s in Env.TERMINAL]
    fig.add_trace(go.Bar(x=[f's={s}' for s in Env.TERMINAL], y=room_counts,
                         marker_color='#9b59b6'),
                  row=1, col=2)

    # Cumulative reward
    fig.add_trace(go.Scatter(x=list(range(len(rewards))),
                             y=np.cumsum(rewards),
                             line=dict(color='#e74c3c')),
                  row=1, col=3)

    fig.update_layout(title=f'{name} — simulated behaviour (80 trials)',
                      showlegend=False, width=800, height=300)
    fig.show()


for name, res in results.items():
    plot_model_detail(name, res)

## Parameter pack/unpack round-trip check

In [ ]:
for name, mod in MODULES.items():
    params = DEFAULT_PARAMS[name]
    x      = pack(params, mod.PARAM_SPEC)
    params2 = unpack(x, mod.PARAM_SPEC)
    ok = all(np.isclose(params[k], params2[k]) for k in params)
    print(f'{name:6s}  pack→unpack round-trip: {"✓" if ok else "FAIL"}')

## RRSF: verify SUD-like vs HC-like behaviour

SUD: high β, low α_SF, high α_p → should show weaker learning and Room 4 bias.  
HC: low β, high α_SF, low α_p → should show stronger goal-directed learning.

In [ ]:
rng_local = np.random.default_rng(99)

params_hc  = {'beta': 1.0,  'gamma': 0.9, 'alpha_sf': 0.5, 'alpha_p': 0.05, 'lam': 0.05}
params_sud = {'beta': 5.0,  'gamma': 0.9, 'alpha_sf': 0.1, 'alpha_p': 0.3,  'lam': 0.05}

acts_hc  = ALL_MODELS['RRSF'].simulate(TRIALS, params_hc,  PI0, np.random.default_rng(1))
acts_sud = ALL_MODELS['RRSF'].simulate(TRIALS, params_sud, PI0, np.random.default_rng(1))

def reward_curve(actions, trials):
    return [Env.reward(Env.step(Env.step(0, a[0]), a[1]), Env.GOALS[g])
            for a, g in zip(actions, trials)]

r_hc  = reward_curve(acts_hc,  TRIALS)
r_sud = reward_curve(acts_sud, TRIALS)

fig = go.Figure()
for label, r, color in [('HC',  r_hc,  '#2196f3'), ('SUD', r_sud, '#f44336')]:
    fig.add_trace(go.Scatter(
        x=list(range(len(smooth(r)))),
        y=smooth(r),
        name=label, line=dict(color=color)
    ))

fig.update_layout(
    title='RRSF: HC-like vs SUD-like parameters (smoothed reward)',
    xaxis_title='Trial', yaxis_title='Reward (moving avg)',
    width=650, height=350,
)
fig.show()

# Room visitation bias
for label, acts in [('HC', acts_hc), ('SUD', acts_sud)]:
    rooms = [Env.step(Env.step(0, a[0]), a[1]) for a in acts]
    dist  = {f's={s}': rooms.count(s) for s in Env.TERMINAL}
    print(f'{label}: {dist}')